# End-to-end workflow for leave-one-out validation

This notebook illustrates how the proposed SC–ST mapping workflow is executed in the context of leave-one-out validation.

The core mapping itself is performed with the standard end-to-end pipeline, including reference preparation, initial mapping, candidate pair selection, directional signal-mixing analysis, pairwise refinement, and final global refinement. Leave-one-out validation is then built on top of this workflow by excluding a target gene from the alignment feature space and evaluating whether its spatial expression can still be reconstructed from the resulting mapping.

The notebook is used to:

1. show the end-to-end SC–ST mapping workflow used for reconstruction,
2. define leave-one-out evaluation settings,
3. reconstruct held-out genes from the resulting mapping,
4. compare predicted and observed spatial expression,
5. assess whether reconstruction remains stable when a gene is excluded from the alignment feature space.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import pickle
import os

import warnings
warnings.filterwarnings('ignore')

# Data Import

In [ ]:
use_sc_data = 'sc' # 'sc' or 'mc'

exclude_st_cell_region = ['YSL', 'Dorsal Forerunner Cells', 'Primordial Germ Cells']

## file path
path_st = '/local/users/mmittnenzweig/spatiotemporal_zebrafish/data/weMerfish/AnnData_withImputation/'
path_mc = '/local/users/dlee/ST/data/timecourse_wt/timecourse_wt.metacells_celltypes-distance-2025-11-19.h5ad'
path_sc = "/local/users/dlee/ST/data/timecourse_wt/make_small_clusters_10_3_100.h5ad"
metacell_graph = '/local/users/dlee/ST/data/cell_type_mgraph.csv'
cell_color_path = '/local/users/dlee/ST/data/cell_type_colors-2025-11-19.csv'

cell_type_color = pd.read_csv(cell_color_path)

In [ ]:
adata_sc = sc.read_h5ad(path_sc)
adata_mc = sc.read_h5ad(path_mc)
adata_st = sc.read_h5ad(path_st + 'B_75p_E1_with_imputation.h5ad')

In [ ]:
st_mask = ~adata_st.obs["clusters"].isin(exclude_st_cell_region)
adata_st = adata_st[st_mask, :].copy()

In [ ]:
adata_sc, _ = mapping_sc_to_st.prep.filter_small_celltypes(
    adata_sc,
    groupby="cell_type",
    min_cells=5,
)

In [ ]:
_ = mapping_sc_to_st.prep.ensure_all_basic(
    adata_sc,
    adata_st,
    st_xy_cols=None,     

    make_spatial_normed=True,   
    spatial_k=10,

    make_scvi=False,            

    make_scvi_st=True,         
    scvi_st_batch_key=None,     
    scvi_st_n_latent=20,
    scvi_st_max_epochs=100,
    scvi_st_counts_layer="counts",
    st_latent_key="X_scvi",
    model_device = 'gpu',
    accelerator = 'gpu',
    devices = [4]
)

In [8]:
adata_sc.layers["counts"] = adata_sc.X.copy()
adata_st.layers["counts"] = adata_st.X.copy()

sc_filter_stat = mapping_sc_to_st.prep.filter_sc_genes_low_counts(
    adata_sc,
    cell_type_key="cell_type",
    layer="counts",
    min_max_expression=0.05,
    min_cells_expressed=0,
    min_cells_per_type = 0,
    inplace=True,
    store_stats=True,
)



common_gene = adata_st.var_names.intersection(adata_sc.var_names)

cc_genes = [
    # DNA replication / S-phase
    "mcm2", "mcm3", "mcm4", "mcm5", "mcm7", "rrm2",
    # G2/M / mitosis
    "mki67", "top2a", "ube2c", "aurkb", "cenpf"
]

filtered_gene = pd.Index(set(common_gene) - set(cc_genes))

adata_sc = adata_sc[:, filtered_gene].copy()
adata_st = adata_st[:, filtered_gene].copy()

[SC gene filter] source=layers['counts']
  genes: 19668 -> 12956 (removed 6712)
  criteria: max_ct_mean_expr>=0.05 AND n_cells_expr>=0
  used_cell_types=28


# Run pipeline and CV

## Configure the leave-one-out pipeline

This cell defines the main pipeline settings used for leave-one-out validation.

These parameters control the mapping, pair selection, directional signal-mixing analysis, and refinement steps that are repeated for each held-out gene. The goal is to keep the pipeline configuration fixed across leave-one-out runs so that any change in reconstruction quality reflects the exclusion of the target gene rather than changes in the analysis settings.

### Parameters

General:
- `n_jobs=128`  
  Number of parallel jobs used during mapping and scoring.

Directional signal-mixing analysis:
- `tau_sc_pos=0`  
  Threshold used to define positive reference-side contrast when identifying directional signal-mixing-associated genes.

- `tau_st_flat=1.1`  
  Threshold used to define weakened spatial contrast for directional signal-mixing analysis.

Candidate pair selection:
- `pair_k_st=20`  
  Number of nearest neighbors used to define ST-side local proximity.

- `pair_k_sc=15`  
  Number of nearest neighbors used to define SC- or metacell-side reference adjacency.

- `pair_threshold_st=20`  
  Minimum support threshold used for ST-derived candidate cell-type pairs.

- `pair_threshold_sc=650`  
  Distance threshold used for reference-side pair selection.

Refinement:
- `solver_epsilon_stage2=0.001`  
  Entropic regularization strength used during pairwise local refinement.

- `solver_epsilon_stage3=0.001`  
  Entropic regularization strength used during the final global refinement.

- `beta_expr_stage3=0.5`  
  Balance between expression-based cost and anchor-based cost during the final global refinement.

### Outputs

- `cfg`  
  Pipeline configuration object reused across leave-one-out runs.

In [17]:
# Configure the pipeline used for leave-one-out validation
cfg = mapping_sc_to_st.run_pipeline.PipelineConfig(
    # Number of parallel jobs used during mapping and scoring
    n_jobs=128,

    # Thresholds for directional signal-mixing analysis
    tau_sc_pos=0,
    tau_st_flat=1.1,

    # Parameters used for candidate pair selection
    pair_k_st=20,
    pair_k_sc=15,
    pair_threshold_st=20,
    pair_threshold_sc=650,

    # Entropic regularization strengths for stage 2 and stage 3 refinement
    solver_epsilon_stage2=0.001,
    solver_epsilon_stage3=0.001,

    # Balance expression cost and anchor-based cost during final refinement
    beta_expr_stage3=0.5
)

## Run the full mapping pipeline

This cell runs the full SC–ST mapping pipeline using the current input datasets and configuration.

The pipeline includes initial mapping, candidate pair selection, directional signal-mixing analysis, pairwise local refinement, and final global refinement. The metacell reference is used to define the reference structure required for pair selection and refinement, while the spatial dataset is updated with the resulting mapping outputs.

### Parameters

Core inputs:
- `adata_sc`  
  Reference dataset used for mapping.

- `adata_st`  
  Spatial dataset used as the alignment target.

- `adata_mc`  
  Metacell reference used to define the stable reference structure.

- `cfg=cfg`  
  Pipeline configuration object controlling mapping, pair selection, and refinement.

Gene selection:
- `selected_genes_by_ct=None`  
  Use the default gene-selection procedure inside the pipeline rather than providing a predefined cell-type-specific gene set.

Reference-graph construction:
- `auto_build_A_df=True`  
  Automatically construct the metacell-based reference adjacency table required for pair selection.

- `recompute_A_df=True`  
  Recompute the reference adjacency table even if a cached version already exists.

- `metacell_graph_path=metacell_graph`  
  Path to the metacell graph used to define the reference structure.

- `cell_type_color=cell_type_color`  
  Cell-type color table used when constructing or visualizing the reference graph.

Other:
- `verbose=True`  
  Print intermediate progress messages during pipeline execution.

### Outputs

- `out`  
  Result dictionary containing intermediate and final outputs from the pipeline run.

- `adata_st`  
  Updated spatial dataset containing the mapping and refinement results.

In [32]:
# Run the full SC–ST mapping pipeline with the current datasets and configuration
out, adata_st = mapping_sc_to_st.run_pipeline.run_pipeline(
    adata_sc,
    adata_st,
    adata_mc,

    # Use the predefined pipeline configuration
    cfg=cfg,

    # Let the pipeline perform its default cell-type-specific gene selection
    selected_genes_by_ct=None,

    # Automatically build the metacell-based reference adjacency table
    auto_build_A_df=True,
    recompute_A_df=True,
    metacell_graph_path=metacell_graph,

    # Provide the cell-type color table for reference-graph construction/visualization
    cell_type_color=cell_type_color,

    # Print progress during execution
    verbose=True,
)

[final_global_anchor_fgw] Step 0/12: Start + validate inputs
[final_global_anchor_fgw]  n_sc=5091, n_st=18019
[final_global_anchor_fgw]  tau_anchor=0.7330532565967884, tau_update=0.9, beta_expr=0.5
[final_global_anchor_fgw]  allow_anchor_update=True
[final_global_anchor_fgw]  prev_weights_key='final_pairwise_weights'
[final_global_anchor_fgw]  alignment: use=True, direction='st_to_sc'
[final_global_anchor_fgw] Step 1/12: Initialize output columns (type snapshot -> working output)
[final_global_anchor_fgw] Step 2/12: Select anchors
[final_global_anchor_fgw]  anchors=656
[final_global_anchor_fgw] Step 3/12: Select update spots
[final_global_anchor_fgw]  updates=18019
[final_global_anchor_fgw] Step 4/12: Define SC candidates
[final_global_anchor_fgw]  sc_candidates=5091
[final_global_anchor_fgw] Step 5/12: Prepare expression features (M_expr built in run_transport)
[final_global_anchor_fgw] Step 7/12: Build pseudo-anchor cost M_anchor from prev weights (cached A-format)
[final_global_anch

## 3. Run LOO Corss Validation

In [11]:
with open('./result/75_E1_selected_genes_set.pkl', 'rb') as f:
    selected_genes_set = pickle.load(f)
    
selected_genes_set = list(selected_genes_set)

In [ ]:
mapping_sc_to_st.run_pipeline.run_loo(
    adata_sc,
    adata_st,
    adata_mc,
    test_gene_list = selected_genes_set,
    save_dir="/local/users/dlee/ST/result/LOO_CV/75%/proposed",
    seed=0,
    cfg = cfg,
    metacell_graph=metacell_graph,
    cell_type_color=cell_type_color,
)

[final_global_anchor_fgw] Step 0/12: Start + validate inputs
[final_global_anchor_fgw]  n_sc=5091, n_st=18019
[final_global_anchor_fgw]  tau_anchor=0.7773447756718812, tau_update=0.9, beta_expr=0.5
[final_global_anchor_fgw]  allow_anchor_update=True
[final_global_anchor_fgw]  prev_weights_key='final_pairwise_weights'
[final_global_anchor_fgw]  alignment: use=True, direction='st_to_sc'
[final_global_anchor_fgw] Step 1/12: Initialize output columns (type snapshot -> working output)
[final_global_anchor_fgw] Step 2/12: Select anchors
[final_global_anchor_fgw]  anchors=676
[final_global_anchor_fgw] Step 3/12: Select update spots
[final_global_anchor_fgw]  updates=18001
[final_global_anchor_fgw] Step 4/12: Define SC candidates
[final_global_anchor_fgw]  sc_candidates=5091
[final_global_anchor_fgw] Step 5/12: Prepare expression features (M_expr built in run_transport)
[final_global_anchor_fgw] Step 7/12: Build pseudo-anchor cost M_anchor from prev weights (cached A-format)
[final_global_anch